# Boussinesq — competitive baselines using their own original prediction protocols

**This is the intended comparison, not the previous K=30 architectural-adaptation notebook.**

**F-FNO**: one-step autoregressive (100 predictions from u0). **U-Net**: one-step autoregressive. **CNO**: directly predicts the field at queried physical time, (u0,t) -> u(t), no rollout. **U-NO-H10**: 10 observed history fields -> next field, 91 recursive predictions. **AWFNO** remains K=30 (its own windowed formulation).

Implemented following the project's Burgers and Cahn–Hilliard baselines and annular geometry adaptations. The original 70 test trajectories and weighted physical annular L2 are unchanged. **U-NO receives 10 observed test fields**, so its full L2 is deliberately not reported; it only has final-time and post-t10 L2 and must be discussed separately from u0-only methods.

## Significant computational cost

The one-step and direct-time baseline training sets now have **100 samples per training trajectory**, rather than just three K30 windows. At N_train=400, batch size 8 and 200 epochs, F-FNO makes **1,000,000 optimizer updates**. Start with one model at N_train=25 before starting the five-size sweep. CUDA memory and runtime can be much higher than in the prior K30 experiments.

This Colab uses readable Python scripts from GitHub, independently saves and resumes checkpoints on Drive, streams training logs, and first runs model-specific architecture shape/gradient checks.

In [ ]:
#@title 1. Drive / GPU / maintained UNO dependency
from google.colab import drive
from pathlib import Path
import os, sys, time, hashlib, urllib.request, subprocess
import torch
drive.mount('/content/gdrive')
assert torch.cuda.is_available(), 'Enable a GPU via Runtime > Change runtime type.'
DATASET=Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
OUTPUT_ROOT=Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
NATIVE_ROOT=OUTPUT_ROOT/'native_reference_2d'
assert DATASET.is_file(), f'Dataset unavailable: {DATASET}'
NATIVE_ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','neuraloperator==2.0.0'],check=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Data GiB:',round(DATASET.stat().st_size/1024**3,3))
print('New outputs:',NATIVE_ROOT)

In [ ]:
#@title 2. Select methods and training sizes (start small)
MODELS='FFNO,CNO,UNet,UNO' #@param {type:'string'}
SAMPLE_SIZES='25' #@param {type:'string'}
SEEDS='1234' #@param {type:'string'}
EPOCHS=200 #@param {type:'integer'}
SMOKE_TEST=True #@param {type:'boolean'}
FORCE_RETRAIN=False #@param {type:'boolean'}
SAVE_FULL_TEST_PREDICTIONS=False #@param {type:'boolean'}
AUTO_DISCONNECT=False #@param {type:'boolean'}
assert set(MODELS.split(',')).issubset({'FFNO','CNO','UNet','UNO'})
assert all(int(n) in (25,50,100,200,400) for n in SAMPLE_SIZES.split(','))
print('Native protocols:',MODELS,'training sizes:',SAMPLE_SIZES,'epochs:',EPOCHS)
print('WARNING: FFNO/CNO/UNet use 100 items per trajectory; UNO uses 91.')

In [ ]:
#@title 3. Download latest readable source with no-cache checks
GITHUB='https://raw.githubusercontent.com/dimigourz/AWFNO/main/Boussinesq2D/'
SOURCE_NAMES=['annulus2d_jcp_experiments.py',
              'reference_architectures_2d.py',
              'native_method_baselines_2d.py']
for filename in SOURCE_NAMES:
    request=urllib.request.Request(GITHUB+filename+'?refresh='+str(time.time_ns()),
                                    headers={'Cache-Control':'no-cache','Pragma':'no-cache'})
    with urllib.request.urlopen(request,timeout=90) as response:
        content=response.read()
    local=Path('/content')/filename
    local.write_bytes(content)
    print(filename,len(content),'bytes SHA256',hashlib.sha256(content).hexdigest())
DRIVER=Path('/content/native_method_baselines_2d.py')
assert DRIVER.is_file()
print('Source:',GITHUB)

In [ ]:
#@title 4. Native-method smoke tests (per architecture)
# Refresh the native architectures even when Cell 3 was not rerun.
# This avoids using the stale U-Net constructor (width instead of channels).
import urllib.request, ast, time
source_url=GITHUB+'reference_architectures_2d.py?refresh='+str(time.time_ns())
req=urllib.request.Request(source_url,headers={'Cache-Control':'no-cache','Pragma':'no-cache'})
with urllib.request.urlopen(req,timeout=90) as response:
    reference_bytes=response.read()
reference_path=Path('/content/reference_architectures_2d.py')
reference_path.write_bytes(reference_bytes)
reference_ast=ast.parse(reference_bytes.decode('utf-8'),filename=str(reference_path))
native_factory=next(
    (node for node in reference_ast.body
     if isinstance(node,ast.FunctionDef) and node.name=='make_native_model'),None)
if native_factory is None:
    raise RuntimeError('The refreshed architecture module has no native model factory.')
unet_calls=[node for node in ast.walk(native_factory)
            if isinstance(node,ast.Call) and isinstance(node.func,ast.Name)
            and node.func.id=='UNet2D']
if len(unet_calls)!=1 or any(k.arg=='width' for k in unet_calls[0].keywords):
    raise RuntimeError('Stale U-Net source: native factory must call UNet2D(..., channels=32), not width=32.')
if not any(k.arg=='channels' and isinstance(k.value,ast.Constant) and k.value.value==32
           for k in unet_calls[0].keywords):
    raise RuntimeError('U-Net constructor mismatch; refusing to run training.')
print('[SOURCE] Refreshed native model architectures; verified UNet2D(..., channels=32).',flush=True)

ENV=os.environ.copy()
ENV.update({
  'DATA_PATH':str(DATASET),'OUTPUT_ROOT':str(OUTPUT_ROOT),
  'NATIVE_OUTPUT':str(NATIVE_ROOT),'NATIVE_MODELS':MODELS,
  'NATIVE_SAMPLE_SIZES':SAMPLE_SIZES,'NATIVE_SEEDS':SEEDS,
  'NATIVE_EPOCHS':str(EPOCHS),
  'NATIVE_FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0',
  'NATIVE_SAVE_ROLLOUTS':'1' if SAVE_FULL_TEST_PREDICTIONS else '0',
  'PYTHONUNBUFFERED':'1',
  'SPLIT_MODE':'contiguous','TIME_SUBSAMPLE':'1',
  'N_VAL':'30','N_TEST':'70',
})
if SMOKE_TEST:
    for model_name in MODELS.split(','):
        print('\n[SMOKE] ',model_name,flush=True)
        env=dict(ENV);env['NATIVE_SMOKE_ONLY']='1';env['NATIVE_MODELS']=model_name
        path=NATIVE_ROOT/f'smoke_{model_name}.log'
        with path.open('w') as output:
            with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                    env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                    bufsize=1,text=True,errors='replace') as proc:
                for line in proc.stdout:
                    print(line,end='',flush=True);output.write(line);output.flush()
                status=proc.wait()
        if status:raise RuntimeError(f'Smoke test failed for {model_name}: {path}')
        print('[SMOKE] PASS',model_name,flush=True)
else:print('Smoke tests disabled.')

In [ ]:
#@title 5. Train the native protocols (logs and checkpoints saved to Drive)
from threading import Thread,Event
logfile=NATIVE_ROOT/'native_training_stdout.log'
stopped=Event()
def heartbeat():
    while not stopped.wait(180):
        print('[HEARTBEAT] Native comparison subprocess alive; last epoch prints show progress.',flush=True)
watcher=Thread(target=heartbeat,daemon=True);watcher.start()
try:
    with logfile.open('a',encoding='utf-8') as log:
        with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                bufsize=1,text=True,errors='replace') as proc:
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            status=proc.wait()
        if status:
            raise RuntimeError(f'Native protocol run exited code {status}. See {logfile}')
finally:
    stopped.set();watcher.join()
print('Completed selected native protocols.',flush=True)

In [ ]:
#@title 6. Read native comparison results
import pandas as pd
from IPython.display import display,Image
p=NATIVE_ROOT/'results.csv'
if p.is_file():
    df=pd.read_csv(p)
    names=['model','N_train','seed','protocol','observed_true_frames',
           'parameters','full_l2_mean','tail_from_t10_l2_mean','final_l2_mean',
           'training_seconds','full_test_set_inference_seconds']
    display(df[names].sort_values(['N_train','model']).reset_index(drop=True))
    print('Do not compare UNO_H10 full L2 against u0-only methods. Its full L2 is undefined.')
else:
    print('No completed model yet; examine the preceding output or',NATIVE_ROOT)
print('Outputs:',NATIVE_ROOT)

In [ ]:
#@title 7. Release GPU only after completion
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()
else:
    print('GPU retained. Disconnect manually after inspecting your numerical results.')